Setup Directories and data files

In [0]:
from pathlib import Path

project_root = Path.cwd()
print("Project Location:", project_root)

In [0]:
data_dir = project_root/"data"
print("Data Location:",data_dir)
print("Data Location Existance:",data_dir.exists())

In [0]:
if data_dir.exists():
    for path in data_dir.glob("*.csv"):
        print(path.name)
        print(path.read_text())

Create DataFrames

In [0]:
schema = "plantId string, temperature double"
observed_df = (
    spark.read.format("csv").schema(schema)
            .option("header","false")
            .load(str(data_dir) + "/plants_observed_temp.csv")
)

required_df = (
    spark.read.format("csv").schema(schema)
            .option("header","false")
            .load(str(data_dir) + "/plants_required_temp.csv")
)


observed_df.show(10)
required_df.show(10)

Average temperture for each plant

In [0]:
from pyspark.sql.functions import *

average_observed_df = observed_df.groupBy("plantId").agg(avg("temperature").alias("avg_temperature"))

In [0]:
average_observed_df.show()

Find Faulty plant

In [0]:
from pyspark.sql.functions import expr

faulty_plants_df = required_df.alias("r").join(average_observed_df.alias("o"),"plantId","inner").withColumn("temp_variance",expr("o.avg_temperature - r.temperature")).withColumn("fault_ind", expr("case when temp_variance >= 5 then 1 else 0 end")).filter("fault_ind = 1").select("r.plantId","o.avg_temperature")

faulty_plants_df.show()




write the datadframe to a location

In [0]:
faulty_plants_df.write.mode("overwrite").format("csv").option("header","true").save("/Volumes/workspace/default/lendingclub/faulty_plants_output")

In [0]:
%sql
select * from read_files('/Volumes/workspace/default/lendingclub/faulty_plants_output',
  format => 'csv',
  header => 'true')

In [0]:
spark.read.format("csv").option("inferSchema",True).option("header","true").load("/Volumes/workspace/default/lendingclub/faulty_plants_output").show()